# ONIA · 1.4 — scikit-learn: de la date brute la selecția modelului

**Notebook didactic extins** · Parcurge celulele în ordine. Conține experimente reproductibile despre stratificare, scalare, codificare, pipeline, matrice de confuzie, validare încrucișată și GridSearchCV.

**Instalare:** `python -m pip install -U numpy pandas matplotlib scikit-learn`.

> Regula de bază: **statisticile transformărilor se învață numai pe train (`fit`); aceleași transformări se aplică atât pe train, cât și pe test (`transform`).** Nu folosi setul final de test pentru alegerea modelului sau a hiperparametrilor.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris, load_diabetes
from sklearn.model_selection import train_test_split

iris=load_iris()
X, y=iris.data, iris.target
print('Caracteristici:',X.shape,'Etichete:',y.shape,'Distribuția claselor:',np.bincount(y))

## 1. Împărțirea datelor. Ce face `stratify`?

`train_test_split` separă rândurile și etichetele corespunzătoare, cu aceeași permutare. `random_state` oferă reproductibilitate. Pentru **clasificare**, `stratify=y` conservă aproximativ proporțiile claselor din vectorul țintă în ambele subseturi.

Dacă utilizăm `stratify=X[:,0]`, conservăm *grupurile din prima caracteristică*. Aceasta nu înseamnă că păstrăm și clasele `y`. Funcționează numai dacă acea coloană reprezintă categorii discrete cu suficient de multe exemple.

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42)
print('Train',X_tr.shape,'etichete',np.bincount(y_tr))
print('Test ',X_te.shape,'etichete',np.bincount(y_te))

fig, axes = plt.subplots(1, 2, figsize=(9,3))
for ax,labels,title in zip(axes,[y_tr,y_te],['Train','Test']):
    ax.bar(np.arange(3),np.bincount(labels),color=['#2e80c4','#e39b44','#54a58d'])
    ax.set_title(title); ax.set_xticks([0,1,2]); ax.set_xlabel('Clasa');ax.set_ylabel('Număr de observații')
plt.tight_layout();plt.show()

In [ ]:
# Exemplu clar: regiune (X[:, 0]) vs clasă (y)
X_region=np.array([[0,i] for i in range(6)] + [[1,i] for i in range(6)])
y_clasa=np.array([0,0,0,0,0,1,0,1,1,1,1,1])

for name,strata in [('y (clasa)',y_clasa),('X[:, 0] (regiune)',X_region[:,0])]:
    a,b,ya,yb=train_test_split(X_region,y_clasa,test_size=1/3,
                               stratify=strata,random_state=42)
    print(name,'→ test: clase',np.bincount(yb,minlength=2),
          'regiuni',np.bincount(b[:,0],minlength=2))

### Clasificare și regresie

În clasificare `y` are valori discrete. În regresie `y` este de regulă continuu, iar folosirea directă a `stratify=y` poate da eroare fiindcă multe valori apar o singură dată. Pentru regresie preferăm de obicei o împărțire simplă, iar dacă e justificat statistic putem stratifica *pe intervale* construite din țintă, cu grijă să existe suficiente observații în fiecare interval.

In [ ]:
X_reg, y_reg = load_diabetes(return_X_y=True)
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg,y_reg,test_size=0.2,random_state=42)
fig, ax=plt.subplots(figsize=(8,3))
ax.hist(y_train_reg, bins=20, alpha=.7,label='Train')
ax.hist(y_test_reg, bins=20, alpha=.7,label='Test')
ax.set_xlabel('Ținta numerică');ax.set_ylabel('Număr');ax.set_title('Regresie: distribuția țintei în train vs test')
ax.legend();plt.tight_layout();plt.show()

## 2. Preprocesare: ce se învață, când se aplică?

- **`fit(X_train)`**: calculează media, deviația, minimul, mediana sau vocabularul categoriilor, după transformator.
- **`transform(X_train)`**: aplică pe train transformarea învățată.
- **`transform(X_test)`**: aplică pe test *aceeași* transformare, fără parametri noi.
- **`fit_transform(X_train)`**: prescurtare pentru antrenare + transformare a train-ului.

Prin urmare, preprocesarea **nu este numai pentru test**: trebuie să primească aceeași reprezentare și modelul la antrenare, și modelul la predicție. Testul se diferențiază prin lipsa lui `fit`.

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
X_train_num=np.array([[2.0,100.0],[4.0,150.0],[6.0,200.0]])
X_test_num=np.array([[3.0,125.0],[9.0,300.0]])

scaler=StandardScaler()
train_scaled=scaler.fit_transform(X_train_num)
test_scaled=scaler.transform(X_test_num)
print('Media train:',scaler.mean_)
print('Abaterea train:',scaler.scale_)
print('Train transformat:\n',train_scaled.round(3))
print('Test transformat:\n',test_scaled.round(3))

In [ ]:
train=np.array([[10.],[20.],[30.]])
test=np.array([[40.]])
minmax=MinMaxScaler()
print('MinMax train:',minmax.fit_transform(train).ravel())
print('MinMax test (poate depăși 1):',minmax.transform(test).ravel())

train_outlier=np.array([[1.],[2.],[3.],[100.]])
robust=RobustScaler()
print('Robust train:',robust.fit_transform(train_outlier).ravel().round(3))
print('Mediană, IQR:',robust.center_,robust.scale_)
print('Robust test 50:',robust.transform([[50.]]).ravel().round(3))

### Vizualizare numerică: aceeași observație, transformări diferite

Comparația se face pe aceleași valori de train și pe un exemplu test, pentru fiecare transformator. Observă că toate metodele pot produce numere în afara domeniului aparent din train când testul are valori extreme.

In [ ]:
values=np.array([[10.],[20.],[30.],[100.]])
new=np.array([[50.]])
fig,axes=plt.subplots(1,3,figsize=(12,3))
for ax,transformer in zip(axes,[StandardScaler(),MinMaxScaler(),RobustScaler()]):
    tr=transformer.fit_transform(values).ravel()
    te=transformer.transform(new).ravel()[0]
    ax.scatter(np.arange(len(tr)),tr,s=75,label='Train')
    ax.scatter([4],[te],marker='*',s=140,label='Test')
    ax.axhline(0,color='#777',linewidth=.8)
    ax.set_title(type(transformer).__name__)
    ax.set_xticks(range(5),['T1','T2','T3','T4','Test'])
    ax.legend()
plt.tight_layout();plt.show()

## 3. Lipsuri și categorii

`SimpleImputer` poate învăța o medie/mediană sau categoria frecventă. `OneHotEncoder` învață categoriile din train și construiește un sistem fix de coloane. Cu `handle_unknown="ignore"` o categorie nouă din test devine un rând de zerouri pentru caracteristica respectivă. `OrdinalEncoder` impune coduri numerice, potrivite doar dacă ordinea este justificată sau gestionată explicit.

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, Normalizer
train_missing=np.array([[1.],[np.nan],[5.]])
test_missing=np.array([[np.nan],[9.]])
imp=SimpleImputer(strategy='mean')
print('Imputer train:',imp.fit_transform(train_missing).ravel())
print('Imputer test:',imp.transform(test_missing).ravel())

encoder=OneHotEncoder(handle_unknown='ignore',sparse_output=False)
train_cat=np.array([['rosu'],['albastru'],['rosu']])
test_cat=np.array([['verde'],['albastru']])
print('Coloane:',encoder.fit(train_cat).get_feature_names_out())
print('Train:\n',encoder.transform(train_cat))
print('Test, inclusiv categorie necunoscută:\n',encoder.transform(test_cat))

In [ ]:
ordinal=OrdinalEncoder(categories=[['mic','mediu','mare']],
                       handle_unknown='use_encoded_value',unknown_value=-1)
print('Ordinal:',ordinal.fit_transform([['mic'],['mare']]).ravel())
print('Ordinal test:',ordinal.transform([['mediu'],['necunoscut']]).ravel())

normalizer=Normalizer(norm='l2')
print('Rând [3,4] →',normalizer.fit_transform([[3,4]]).round(3))
print('Rând [6,8] →',normalizer.transform([[6,8]]).round(3))

## 4. Pipeline: un singur obiect, două comportamente

La `pipeline.fit(X_train,y_train)`: **scaler.fit_transform(X_train)**, apoi **classifier.fit(...)**.

La `pipeline.predict(X_test)`: **scaler.transform(X_test)**, apoi **classifier.predict(...)**. Parametrii nu se schimbă în faza de predicție.

`make_pipeline` generează nume de pași (`standardscaler`, `logisticregression`). `Pipeline([...])` îți permite să alegi nume personalizate (`scalare`, `model`).

In [ ]:
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

X, y=load_iris(return_X_y=True)
X_train, X_test, y_train, y_test=train_test_split(X,y,test_size=.25,
                                                stratify=y,random_state=42)
model=make_pipeline(StandardScaler(),LogisticRegression(max_iter=600))
model.fit(X_train,y_train)
pred=model.predict(X_test)
print('Acuratețe:',accuracy_score(y_test,pred))
print('Nume pași:',list(model.named_steps))
print('Medii scaler, învățate din train:',model.named_steps['standardscaler'].mean_)

In [ ]:
from sklearn.svm import SVC
model_explicit=Pipeline([
    ('scalare',StandardScaler()),
    ('clasificator',SVC(C=1.0,kernel='rbf'))
])
model_explicit.fit(X_train,y_train)
print('C=',model_explicit.get_params()['clasificator__C'])
print('Primele 3 predicții:',model_explicit.predict(X_test[:3]))

### Date mixte: `ColumnTransformer`

Pentru un tabel în care unele coloane sunt numere (posibil cu lipsuri), iar altele sunt categorice, folosim transformări diferite pe grupuri de coloane și le reunim înainte de clasificator.

In [ ]:
from sklearn.compose import ColumnTransformer

frame=pd.DataFrame({'pret':[10.,15.,np.nan,22.,30.,12.,27.,8.],
                    'livrare_zile':[2.,5.,1.,np.nan,4.,2.,7.,1.],
                    'categorie':['A','B','A','C','B','A','C','B']})
target=np.array([0,1,0,1,1,0,1,0])

numeric=make_pipeline(SimpleImputer(strategy='median'),StandardScaler())
categoric=make_pipeline(SimpleImputer(strategy='most_frequent'),
                        OneHotEncoder(handle_unknown='ignore'))
col=ColumnTransformer([('num',numeric,['pret','livrare_zile']),
                       ('cat',categoric,['categorie'])])
pipe=make_pipeline(col,LogisticRegression(max_iter=600))
pipe.fit(frame,target)
new=pd.DataFrame({'pret':[17.], 'livrare_zile':[3.], 'categorie':['D']})
print('Predicția unei categorii noi:',pipe.predict(new))

## 5. Matricea de confuzie: rând = real, coloană = prezis

Pentru `reale = [0,1,1,0,1,0]` și `prezise = [0,1,0,0,1,1]`, matricea este `[[2,1],[1,2]]`: două predicții corecte pentru clasa 0, două pentru clasa 1, o confuzie în fiecare sens. Dacă clasa 1 este pozitivă: TN=2, FP=1, FN=1, TP=2.

Diagonală = corect. În afara diagonalei = eroare. Culoarea mai intensă indică de regulă un număr/procent mai mare, nu neapărat o decizie mai bună. Normalizarea `normalize='true'` împarte fiecare rând la totalul clasei reale.

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
reale=[0,1,1,0,1,0]
prezise=[0,1,0,0,1,1]
print('Acuratețe:',accuracy_score(reale,prezise))
print('Matrice:\n',confusion_matrix(reale,prezise,labels=[0,1]))
ConfusionMatrixDisplay.from_predictions(reale,prezise,labels=[0,1],
                                       display_labels=['Clasa 0','Clasa 1'],
                                       cmap='Blues',values_format='d',colorbar=True)
plt.title('Rând = real ↓, coloană = prezis →')
plt.tight_layout();plt.show()

In [ ]:
# Iris: mai multe clase și normalizare pe rânduri
mod=make_pipeline(StandardScaler(),LogisticRegression(max_iter=600))
mod.fit(X_train,y_train)
fig, ax=plt.subplots(figsize=(6,5))
ConfusionMatrixDisplay.from_predictions(y_test,mod.predict(X_test),
                                       display_labels=iris.target_names,
                                       normalize='true',cmap='Blues',values_format='.2f',ax=ax)
plt.tight_layout();plt.show()

## 6. Validare încrucișată: **antrenare de la zero pe fiecare fold**

`cross_val_score` împarte datele de dezvoltare în K fold-uri și **clonează un estimator neantrenat pentru fiecare split**. În fiecare iterație, preprocesarea din pipeline își reînvață statisticile doar din fold-urile train; fold-ul validare primește numai `transform` și `predict`.

Cu `cv=5`: cinci scoruri de validare pentru o configurație, nu cinci modele finale. Pentru clasificare stratificată folosim `StratifiedKFold`, pentru regresie `KFold`; pentru serii temporale alegem protocol temporal adecvat.

In [ ]:
from sklearn.model_selection import StratifiedKFold, KFold, cross_val_score
from sklearn.base import clone

X,y=load_iris(return_X_y=True)
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
model=make_pipeline(StandardScaler(),LogisticRegression(max_iter=600))
print('Scoruri CV:',cross_val_score(model,X,y,cv=cv,scoring='accuracy').round(3))
for fold,(idx_tr,idx_val) in enumerate(cv.split(X,y),1):
    copie=clone(model)  # neantrenată, cu hiperparametrii originali
    copie.fit(X[idx_tr],y[idx_tr])
    print('Fold',fold,'| train',len(idx_tr),'| validare',len(idx_val),
          '| acuratețe',round(copie.score(X[idx_val],y[idx_val]),3))

In [ ]:
X_reg,y_reg=load_diabetes(return_X_y=True)
reg=make_pipeline(StandardScaler(),__import__('sklearn.linear_model',fromlist=['Ridge']).Ridge(alpha=1.0))
cv_reg=KFold(n_splits=5,shuffle=True,random_state=42)
neg_mae=cross_val_score(reg,X_reg,y_reg,cv=cv_reg,scoring='neg_mean_absolute_error')
print('MAE per fold:',(-neg_mae).round(2),'MAE mediu:',round(-neg_mae.mean(),2))

### Olimpiadă vs. proiect real

**În competiție**: date etichetate pentru antrenare + test oficial fără etichete. Folosești CV sau o validare locală desprinsă din datele etichetate, alegi modelul, apoi îl reantrenezi pe **toate datele etichetate** și generezi predicții pentru testul oficial. Nu poți calcula local metrici fără etichetele oficiale.

**În aplicații reale**: păstrezi un test final etichetat, complet neatins pentru comparație. Alegerea modelului se face prin CV pe train. Apoi modelul selectat este reantrenat pe toate datele de dezvoltare și evaluat o singură dată pe test; poți antrena ulterior un model de producție cu mai multe date, însă acela nu mai are testul anterior drept evaluare independentă.

In [ ]:
# Demonstrația completă a unui test oficial cu etichete ascunse.
X,y=load_iris(return_X_y=True)
X_labeled,X_test_official,y_labeled,_=train_test_split(
    X,y,test_size=.2,random_state=42,stratify=y)
X_tr,X_val,y_tr,y_val=train_test_split(
    X_labeled,y_labeled,test_size=.2,random_state=42,stratify=y_labeled)
model_candidat=make_pipeline(StandardScaler(),LogisticRegression(max_iter=600))
model_candidat.fit(X_tr,y_tr)
print('Scor local pe validare:',model_candidat.score(X_val,y_val))
model_final=clone(model_candidat)
model_final.fit(X_labeled,y_labeled)
predictii_oficiale=model_final.predict(X_test_official)
print('Predicții oficiale (fără a consulta etichetele ascunse):',predictii_oficiale[:10])

## 7. GridSearchCV: fiecare combinație × toate fold-urile

Pentru `n_neighbors in [1,3,5]` și `metric in ['euclidean','manhattan']` avem **3 × 2 = 6 configurații**. Cu `cv=5` sunt **30 de antrenări** pentru evaluare. `refit=True` (implicit) adaugă un antrenament al configurației câștigătoare pe întregul set transmis lui `fit`.

- `best_params_`: hiperparametrii configurației cu cel mai bun scor mediu CV.
- `best_score_`: media scorurilor de validare pentru câștigător, **nu testul final**.
- `best_estimator_`: estimatorul refăcut după selecție pe toate datele de dezvoltare (când `refit=True`).
- `cv_results_`: toate combinațiile și scorurile pe fold-uri.

`cv=5` înseamnă *numărul de împărțiri pentru validare*, nu epoci, clase sau modele finale.

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

X,y=load_iris(return_X_y=True)
X_tr,X_te,y_tr,y_te=train_test_split(X,y,test_size=.2,
                                    stratify=y,random_state=42)
pipe=make_pipeline(StandardScaler(),KNeighborsClassifier())
grid={'kneighborsclassifier__n_neighbors':[1,3,5],
      'kneighborsclassifier__metric':['euclidean','manhattan']}
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
search=GridSearchCV(pipe,grid,cv=cv,scoring='accuracy',refit=True)
search.fit(X_tr,y_tr)  # doar dezvoltare, NICIODATĂ testul final
print('best_params_:',search.best_params_)
print('best_score_ (CV):',round(search.best_score_,4))
print('best_estimator_:',search.best_estimator_)
print('scor test final:',round(accuracy_score(y_te,search.predict(X_te)),4))

In [ ]:
rezultate=pd.DataFrame(search.cv_results_)
coloane=['param_kneighborsclassifier__n_neighbors',
         'param_kneighborsclassifier__metric',
         'mean_test_score','std_test_score','rank_test_score']
print(rezultate[coloane].sort_values('rank_test_score').to_string(index=False))
# mean_test_score = media pe validările cross-validation; nu e X_te!

In [ ]:
# SVM: alt set de hiperparametri și altă metrică
from sklearn.svm import SVC
svc=make_pipeline(StandardScaler(),SVC(kernel='rbf'))
search_svc=GridSearchCV(svc,{'svc__C':[0.1,1,10],
                             'svc__gamma':['scale',0.01,0.1]},
                        cv=5,scoring='f1_macro')
search_svc.fit(X_tr,y_tr)
print('SVM best_params_:',search_svc.best_params_)
print('SVM best_score_ (f1_macro CV):',round(search_svc.best_score_,3))

In [ ]:
# Regresie: alpha la Ridge, scorul negativ MAE (mai mare = mai bine)
from sklearn.linear_model import Ridge
X_reg,y_reg=load_diabetes(return_X_y=True)
reg=make_pipeline(StandardScaler(),Ridge())
search_reg=GridSearchCV(reg,{'ridge__alpha':[0.01,.1,1,10,100]},
                        cv=KFold(5,shuffle=True,random_state=42),
                        scoring='neg_mean_absolute_error')
search_reg.fit(X_reg,y_reg)
print('Ridge best_params_:',search_reg.best_params_)
print('MAE CV pozitiv:',round(-search_reg.best_score_,3))

## 8. Exerciții de verificare

1. Modifică valoarea `100` din test la `1000` în exemplul MinMaxScaler. De ce nu se schimbă minimul/maximul învățat din train?
2. Transformă 5 tipuri de produse prin OneHotEncoder, apoi adaugă o categorie nouă numai în test. Ce se întâmplă cu numărul de coloane?
3. Înlocuiește `StratifiedKFold(5)` cu `StratifiedKFold(3)` și explică de ce numărul de antrenări din GridSearch scade.
4. Compară `scoring='accuracy'` cu `scoring='f1_macro'` pentru aceeași grilă. Pot diferi `best_params_`?
5. Explică de ce **nu** este corect `StandardScaler().fit_transform(X_train+X_test)` și de ce `fit_transform(X_test)` este diferit de `transform(X_test)`.

**Pentru studiu suplimentar**: [Prevenirea data leakage](https://scikit-learn.org/stable/common_pitfalls.html), [GridSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html), [Cross-validation](https://scikit-learn.org/stable/modules/cross_validation.html).